# E1 vs E3: project cross-dataset / zero-shot stress test
Train and calibrate on CelebA-Spoof only. LCC-FASD image-level; CASIA-FASD video/session mean score. No target tuning. This notebook requires explicit distribution provenance and label mapping. E2 diagnostic is omitted to keep the primary comparison focused.

In [1]:
%pip install -q albumentations onnxruntime scikit-learn pandas psutil insightface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 39.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, glob, json, time, math, random, hashlib, shutil, sys, platform
from pathlib import Path, PurePosixPath
from collections import Counter
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
import albumentations as A
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve
import onnxruntime as ort
import psutil
SAMPLE_SEED = 42
INPUT_SIZE = 224
FREQ_DIM = 64
BBOX_JITTER_P = 0.20
BBOX_JITTER_SCALE = (0.95, 1.05)
BBOX_JITTER_TRANSLATE = 0.05
DCT_EPS = 1e-6
CELEBA_MEAN = [0.5931, 0.4690, 0.4229]
CELEBA_STD = [0.2471, 0.2214, 0.2157]
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
print({'python':sys.version, 'torch':torch.__version__, 'opencv':cv2.__version__, 'ort':ort.__version__, 'albumentations':A.__version__})

{'python': '3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]', 'torch': '2.10.0+cu128', 'opencv': '4.13.0', 'ort': '1.30.0', 'albumentations': '2.0.8'}


## Configure mounted datasets, CSV mappings, frozen models and detector

In [3]:
import pandas as pd
from pathlib import Path

# Đọc danh sách ảnh từ 2 file text có sẵn của dataset
lcc_root = Path("/kaggle/input/datasets/aleksandrpikul222/lcc-fasd/LCC_FASD")
real_files = (lcc_root / "CLIENT_TEST.txt").read_text().strip().splitlines()
spoof_files = (lcc_root / "IMPOSTER_TEST.txt").read_text().strip().splitlines()

rows = []
for f in real_files:
    if f.strip():
        rows.append({"path": f.strip(), "label": 0, "partition": "evaluation"})
for f in spoof_files:
    if f.strip():
        rows.append({"path": f.strip(), "label": 1, "partition": "evaluation"})

manifest_path = "/kaggle/working/lcc_manifest.csv"
pd.DataFrame(rows).to_csv(manifest_path, index=False)
print(f"Đã tạo manifest thành công: {len(rows)} ảnh (Real: {len(real_files)}, Spoof: {len(spoof_files)})")


Đã tạo manifest thành công: 3766 ảnh (Real: 389, Spoof: 3377)


In [4]:
import pandas as pd
from pathlib import Path

E1_RUN_MODE = "preliminary"
E1_ARTIFACT_DIR = "/kaggle/input/datasets/locngodev/e1-v5-3-artifacts/E1_v5_3_mnv3_small"
E3_ARTIFACT_DIR = "/kaggle/input/datasets/locngodev/e3-spatial-frequency-concat/E3_concat_v1"
LCC_FASD_ROOT = "/kaggle/input/datasets/aleksandrpikul222/lcc-fasd/LCC_FASD"
CASIA_FASD_ROOT = ""
REPLAY_ATTACK_ROOT = ""  # optional
SCRFD_MODEL_PATH = "/kaggle/input/datasets/locngodev/scrfd-model/det_500m.onnx"
EXTERNAL_CACHE_INPUT_PATH = ""
FRAMES_PER_VIDEO = 16
FRAME_SAMPLE_SEED = 42
MIN_VALID_FRAMES_PER_VIDEO = 1

# --- TỰ ĐỘNG TẠO MANIFEST CHO LCC-FASD TỪ FILE TXT CÓ SẴN ---
lcc_root = Path(LCC_FASD_ROOT)
real_lines = (lcc_root / "CLIENT_TEST.txt").read_text().strip().splitlines()
spoof_lines = (lcc_root / "IMPOSTER_TEST.txt").read_text().strip().splitlines()

rows = []
for p in real_lines:
    if p.strip():
        # Lấy tên file gốc (tránh trường hợp file txt chứa đường dẫn con)
        fname = Path(p.strip()).name
        rows.append({"path": fname, "label": 0, "partition": "evaluation"})
for p in spoof_lines:
    if p.strip():
        fname = Path(p.strip()).name
        rows.append({"path": fname, "label": 1, "partition": "evaluation"})

manifest_csv_path = "/kaggle/working/lcc_manifest.csv"
pd.DataFrame(rows).to_csv(manifest_csv_path, index=False)
print(f"✅ Đã tạo lcc_manifest.csv thành công với {len(rows)} mẫu (Real: {len(real_lines)}, Spoof: {len(spoof_lines)})")

# --- CẤU HÌNH DATASET_CONFIG ---
DATASET_CONFIG = {
 'lcc_fasd': {
     'kind': 'image',
     'partition': 'evaluation',
     'manifest_csv': manifest_csv_path,  # <-- Tự động nạp file CSV vừa tạo
     'folder_mapping': {},
     'identity': 'LCC-FASD',
     'version': '1.0',
     'source': 'aleksandrpikul222/lcc-fasd',
     'license': 'Academic/Research',
     'provenance_confirmed': True
 },
 'casia_fasd': {
     'kind': 'video',
     'partition': 'test',
     'manifest_csv': '',
     'folder_mapping': {},
     'identity': 'CASIA-FASD',
     'version': '1.0',
     'source': 'CASIA',
     'license': 'Academic/Research',
     'provenance_confirmed': True
 },
 'replay_attack': {
     'kind': 'video',
     'partition': 'test',
     'manifest_csv': '',
     'folder_mapping': {},
     'identity': 'Replay-Attack',
     'version': '1.0',
     'source': 'Idiap',
     'license': 'Academic/Research',
     'provenance_confirmed': True
 }
}

LICENSE_METADATA_PATHS = []
E1_INDOMAIN_SUMMARY = ""
E3_INDOMAIN_SUMMARY = ""
RESULT_DIR = Path('/kaggle/working') / f'cross_dataset_{E1_RUN_MODE}_v1'
RESULT_DIR.mkdir(parents=True, exist_ok=True)


✅ Đã tạo lcc_manifest.csv thành công với 3766 mẫu (Real: 389, Spoof: 3377)


In [5]:
def find_exact_artifact(filename, explicit_dir=""):
    if explicit_dir:
        if not explicit_dir.startswith("/"):
            explicit_dir = "/" + explicit_dir
        p = os.path.join(explicit_dir, filename)
        if os.path.exists(p):
            return p
        # Tự động quét tìm trong các thư mục con data_protocol, deployment, metadata
        sub_matches = sorted(set(glob.glob(os.path.join(explicit_dir, "**", filename), recursive=True)))
        if len(sub_matches) == 1:
            return sub_matches[0]
        if len(sub_matches) > 1:
            raise RuntimeError(f"Multiple copies of {filename} found under {explicit_dir}:\n" + "\n".join(sub_matches))
        raise FileNotFoundError(f"Missing required artifact {filename} under {explicit_dir}")

    matches = glob.glob(os.path.join("/kaggle/input", "**", filename), recursive=True)
    matches = sorted(set(matches))
    if len(matches) == 0:
        raise FileNotFoundError(f"Could not find {filename} under /kaggle/input.")
    if len(matches) > 1:
        raise RuntimeError(f"Multiple copies of {filename} found:\n" + "\n".join(matches))
    return matches[0]

def find_optional_artifact(filename, explicit_dir=""):
    try:
        return find_exact_artifact(filename, explicit_dir)
    except FileNotFoundError:
        return None

def label_of(meta_value):
    if len(meta_value) <= 40:
        raise ValueError(f"Metadata entry too short: len={len(meta_value)}")
    raw = int(meta_value[40])
    if raw == 0:
        return 0
    if raw in [1, 2, 3, 4, 5, 6, 7]:
        return 1
    return 2

def class_counts(keys, meta):
    y = np.array([label_of(meta[str(k)]) for k in keys], dtype=np.int64)
    return {c: int((y == c).sum()) for c in [0, 1, 2]}

def split_fingerprint(keys):
    payload = "\n".join(sorted(map(str, keys))).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

def selected_cache_fingerprint(keys, records):
    h = hashlib.sha256()
    for key in sorted(map(str, keys)):
        h.update(key.encode("utf-8"))
        h.update(b"\0")
        h.update(
            json.dumps(
                records[key],
                sort_keys=True,
                separators=(",", ":"),
            ).encode("utf-8")
        )
        h.update(b"\n")
    return h.hexdigest()

def runtime_crop_bgr(img, bbox_xyxy, bbox_expansion_factor):
    original_height, original_width = img.shape[:2]
    x1, y1, x2, y2 = map(float, bbox_xyxy)
    w = x2 - x1
    h = y2 - y1
    if w <= 0 or h <= 0:
        raise ValueError(f"Invalid xyxy bbox: {bbox_xyxy}")

    max_dim = max(w, h)
    center_x = x1 + w / 2.0
    center_y = y1 + h / 2.0

    x_start = int(center_x - max_dim * bbox_expansion_factor / 2.0)
    y_start = int(center_y - max_dim * bbox_expansion_factor / 2.0)
    crop_size = int(max_dim * bbox_expansion_factor)
    if crop_size <= 0:
        raise ValueError(f"Invalid crop_size={crop_size}")

    crop_x1 = max(0, x_start)
    crop_y1 = max(0, y_start)
    crop_x2 = min(original_width, x_start + crop_size)
    crop_y2 = min(original_height, y_start + crop_size)

    top_pad = int(max(0, -y_start))
    left_pad = int(max(0, -x_start))
    bottom_pad = int(max(0, (y_start + crop_size) - original_height))
    right_pad = int(max(0, (x_start + crop_size) - original_width))

    if crop_x2 <= crop_x1 or crop_y2 <= crop_y1:
        raise RuntimeError(f"Empty crop intersection for bbox={bbox_xyxy}")

    cropped_img = img[crop_y1:crop_y2, crop_x1:crop_x2, :]
    result = cv2.copyMakeBorder(
        cropped_img,
        top_pad,
        bottom_pad,
        left_pad,
        right_pad,
        cv2.BORDER_REFLECT_101,
    )

    if result.shape[0] != crop_size or result.shape[1] != crop_size:
        result = cv2.resize(
            result,
            (crop_size, crop_size),
            interpolation=cv2.INTER_AREA,
        )
    return result

def read_cached_face_crop_rgb(
    img_path,
    cache_record,
    bbox_jitter=False,
    bbox_jitter_p=BBOX_JITTER_P,
    jitter_scale=BBOX_JITTER_SCALE,
    jitter_translate=BBOX_JITTER_TRANSLATE,
):
    image_bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise FileNotFoundError(f"Cannot read image: {img_path}")

    if cache_record.get("status") == "INVALID":
        raise RuntimeError(
            f"Invalid cached preprocessing record for {img_path}: {cache_record}"
        )

    x1, y1, x2, y2 = map(float, cache_record["bbox_xyxy"])
    w = x2 - x1
    h = y2 - y1
    if w <= 0 or h <= 0:
        raise ValueError(f"Non-positive cached bbox for {img_path}")

    if bbox_jitter and random.random() < float(bbox_jitter_p):
        base = max(w, h)
        cx = x1 + w / 2.0
        cy = y1 + h / 2.0

        scale = random.uniform(
            float(jitter_scale[0]),
            float(jitter_scale[1]),
        )
        cx += random.uniform(-jitter_translate, jitter_translate) * base
        cy += random.uniform(-jitter_translate, jitter_translate) * base
        w *= scale
        h *= scale

        x1 = cx - w / 2.0
        y1 = cy - h / 2.0
        x2 = x1 + w
        y2 = y1 + h

    crop_bgr = runtime_crop_bgr(
        image_bgr,
        (x1, y1, x2, y2),
        bbox_expansion_factor=float(cache_record["crop_factor"]),
    )
    return cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)

def resize_rgb_to_input(img_rgb, size=INPUT_SIZE):
    old_h, old_w = img_rgb.shape[:2]
    ratio = float(size) / max(old_h, old_w)
    scaled_h = max(1, int(old_h * ratio))
    scaled_w = max(1, int(old_w * ratio))
    interpolation = cv2.INTER_LANCZOS4 if ratio > 1.0 else cv2.INTER_AREA

    img = cv2.resize(
        img_rgb,
        (scaled_w, scaled_h),
        interpolation=interpolation,
    )

    delta_w = size - scaled_w
    delta_h = size - scaled_h
    top, bottom = delta_h // 2, delta_h - delta_h // 2
    left, right = delta_w // 2, delta_w - delta_w // 2

    return cv2.copyMakeBorder(
        img,
        top,
        bottom,
        left,
        right,
        cv2.BORDER_REFLECT_101,
    )

def rgb_to_luminance_float(img_rgb):
    rgb = img_rgb.astype(np.float32) / 255.0
    # Standard RGB luminance weighting.
    return (
        0.299 * rgb[..., 0]
        + 0.587 * rgb[..., 1]
        + 0.114 * rgb[..., 2]
    ).astype(np.float32)

def luminance_to_dct_map(luma):
    if luma.shape != (INPUT_SIZE, INPUT_SIZE):
        raise ValueError(f"Expected {(INPUT_SIZE, INPUT_SIZE)}, got {luma.shape}")

    coeff = cv2.dct(np.ascontiguousarray(luma, dtype=np.float32))
    coeff = np.sign(coeff) * np.log1p(np.abs(coeff))

    mean = float(coeff.mean())
    std = float(coeff.std())
    coeff = (coeff - mean) / max(std, DCT_EPS)

    return coeff[None, ...].astype(np.float32)

def rgb_to_frequency_tensor(img_rgb):
    img_rgb = resize_rgb_to_input(img_rgb, INPUT_SIZE)
    luma = rgb_to_luminance_float(img_rgb)
    dct_map = luminance_to_dct_map(luma)
    return torch.from_numpy(np.ascontiguousarray(dct_map))

def runtime_preprocess_rgb(img_rgb, model_img_size, mean, std):
    """Match runtime resize + REFLECT_101 letterbox + normalize + NCHW."""
    if img_rgb is None or img_rgb.size == 0:
        raise ValueError("Cannot preprocess an empty image")

    new_size = int(model_img_size)
    old_size = img_rgb.shape[:2]
    ratio = float(new_size) / max(old_size)
    scaled_shape = tuple(max(1, int(x * ratio)) for x in old_size)
    interpolation = cv2.INTER_LANCZOS4 if ratio > 1.0 else cv2.INTER_AREA
    img = cv2.resize(img_rgb, (scaled_shape[1], scaled_shape[0]), interpolation=interpolation)

    delta_w = new_size - scaled_shape[1]
    delta_h = new_size - scaled_shape[0]
    top, bottom = delta_h // 2, delta_h - (delta_h // 2)
    left, right = delta_w // 2, delta_w - (delta_w // 2)
    img = cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_REFLECT_101)

    arr = img.transpose(2, 0, 1).astype(np.float32) / 255.0
    if mean is not None and std is not None:
        mean_arr = np.asarray(mean, dtype=np.float32).reshape(3, 1, 1)
        std_arr = np.asarray(std, dtype=np.float32).reshape(3, 1, 1)
        arr = (arr - mean_arr) / std_arr
    return torch.from_numpy(np.ascontiguousarray(arr))

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()

def json_clean(x):
    if isinstance(x, dict): return {str(k):json_clean(v) for k,v in x.items()}
    if isinstance(x, (list,tuple)): return [json_clean(v) for v in x]
    if isinstance(x, np.ndarray): return json_clean(x.tolist())
    if isinstance(x, (np.integer,)): return int(x)
    if isinstance(x, (float,np.floating)): return float(x) if np.isfinite(x) else None
    return x

def save_json(path, value):
    Path(path).write_text(json.dumps(json_clean(value),indent=2,allow_nan=False))

def branch_inputs(crop_rgb):
    # One shared augmented crop; exact E1 spatial and E2 DCT functions.
    rgb = runtime_preprocess_rgb(crop_rgb, 224, CELEBA_MEAN, CELEBA_STD)
    freq = rgb_to_frequency_tensor(crop_rgb)
    return rgb, freq

def verify_disjoint(splits):
    sets = [set(map(str,k)) for k in splits]
    if any(len(s)!=len(k) for s,k in zip(sets,splits)): raise RuntimeError('Duplicate split keys')
    if any(sets[i]&sets[j] for i in range(3) for j in range(i+1,3)): raise RuntimeError('Train/Val/Test overlap')
    subjects = [{PurePosixPath(k).parts[2] for k in s} for s in sets]
    if any(subjects[i]&subjects[j] for i in range(3) for j in range(i+1,3)): raise RuntimeError('Subject overlap')

DCT_SPEC = {'implementation':'cv2.dct float32 orthonormal 2D', 'luminance_weights_rgb':[0.299,0.587,0.114],
 'rgb_range':'uint8 -> float32 / 255', 'dynamic_range':'sign(C)*log1p(abs(C))',
 'normalization':'per-sample z-score over full map, population std', 'epsilon':1e-6, 'mask':'none',
 'resize':'longest side 224; LANCZOS4 upscale, AREA downscale; REFLECT_101 letterbox'}


def binary_pad_score_from_logits(logits_np):
    logits = np.asarray(logits_np, dtype=np.float64)
    real = logits[:, 0]
    spoof = logits[:, 1:]

    m = np.max(spoof, axis=1)
    spoof_lse = m + np.log(
        np.exp(spoof[:, 0] - m)
        + np.exp(spoof[:, 1] - m)
    )
    return real - spoof_lse

def metrics_from_scores(y3, d, threshold):
    y3 = np.asarray(y3, dtype=np.int64)
    d = np.asarray(d, dtype=np.float64)

    is_real = (y3 == 0)
    is_attack = ~is_real
    pred_real = d >= threshold

    bpcer = (
        float(np.mean(~pred_real[is_real]))
        if np.any(is_real) else float("nan")
    )
    apcer = (
        float(np.mean(pred_real[is_attack]))
        if np.any(is_attack) else float("nan")
    )
    acer = 0.5 * (apcer + bpcer)

    y_binary = is_real.astype(np.int64)
    auc = float(roc_auc_score(y_binary, d)) if np.unique(y_binary).size==2 else float('nan')

    accuracy = float(np.mean(pred_real == is_real))
    cm = confusion_matrix(
        y_binary,
        pred_real.astype(np.int64),
        labels=[0, 1],
    ).tolist()

    return {
        "Accuracy": accuracy,
        "APCER": apcer,
        "BPCER": bpcer,
        "ACER": acer,
        "AUC": auc,
        "confusion_matrix": cm,
    }

def sigmoid(x):
    return float(1/(1+np.exp(-np.clip(x,-700,700))))

def pad_metrics(y, d, threshold):
    y,d = np.asarray(y),np.asarray(d,dtype=float)
    if len(y)==0:
        return {'N':0,'N_real':0,'N_attack':0,'AUC':None,'HTER':None,'APCER':None,'BPCER':None,'ACER':None,'Accuracy':None}
    if len(y)!=len(d) or not np.isfinite(d).all(): raise RuntimeError('Invalid scores')
    m=metrics_from_scores(y,d,threshold)
    m.update(N=len(y),N_real=int((y==0).sum()),N_attack=int((y!=0).sum()),HTER=m['ACER'])
    return json_clean(m)

FACE_BINS = [-np.inf,48,100,140,180,np.inf]
FACE_LABELS = ['<48','48–99','100–139','140–179','>=180']
def face_size_breakdown(df, threshold):
    rows=[]
    bins=pd.cut(df.face_min_side, FACE_BINS, labels=FACE_LABELS, right=False)
    for name in FACE_LABELS:
        g=df[bins==name]
        rows.append({'face_size_bin':name,**pad_metrics(g.true_class,g.d,threshold)})
    return pd.DataFrame(rows)

def save_plots(df, threshold, prefix):
    fig,ax=plt.subplots()
    for label,name in [(0,'Real'),(1,'Attack')]:
        vals=df.loc[(df.true_class!=0).astype(int)==label,'d']
        if len(vals): ax.hist(vals,bins=40,alpha=.5,label=name)
    ax.axvline(threshold,linestyle='--',label='Source Validation threshold');ax.legend();ax.set_xlabel('d')
    fig.savefig(str(prefix)+'_score_distribution.png',bbox_inches='tight');plt.close(fig)
    fig,ax=plt.subplots()
    if df.true_class.eq(0).any() and df.true_class.ne(0).any():
        fpr,tpr,_=roc_curve(df.true_class.eq(0).astype(int),df.d);ax.plot(fpr,tpr)
    ax.set(xlabel='Attack acceptance rate',ylabel='Real acceptance rate')
    fig.savefig(str(prefix)+'_roc.png',bbox_inches='tight');plt.close(fig)

ORT_THREADS = 2
def cpu_session(path):
    opts=ort.SessionOptions();opts.intra_op_num_threads=ORT_THREADS;opts.inter_op_num_threads=1
    return ort.InferenceSession(str(path),sess_options=opts,providers=['CPUExecutionProvider'])

def load_frozen_model(kind, directory, mode):
    names={'E1':f'mnv3s_e1_{mode}_v5_3_edge','E3':f'e3_concat_{mode}_v1'}
    stem=names[kind]
    onnx_path=find_exact_artifact(stem+'_best.onnx',directory)
    config=json.loads(Path(find_exact_artifact(stem+'_runtime_config.json',directory)).read_text())
    meta=json.loads(Path(find_exact_artifact(stem+'_best_meta.json',directory)).read_text())
    thresholds=[float(x['calibrated_logit_threshold']) for x in [config,meta]]
    if not np.isfinite(thresholds).all() or abs(thresholds[0]-thresholds[1])>1e-9: raise RuntimeError('Conflicting locked thresholds')
    for x in [config,meta]:
        if not np.isfinite(float(x.get('calibrated_probability_threshold',x.get('predictor_threshold_probability',float('nan'))))) or abs(float(x.get('calibrated_probability_threshold',x.get('predictor_threshold_probability',float('nan'))))-sigmoid(thresholds[0]))>1e-7: raise RuntimeError('Probability threshold conflict')
    if config.get('onnx_sha256') and config['onnx_sha256']!=sha256_file(onnx_path): raise RuntimeError('ONNX hash mismatch')
    if kind=='E3':
        if config['e1_run_mode']!=mode or meta['e1_run_mode']!=mode: raise RuntimeError('Run-mode mismatch')
        if config['init_mode']!='independent': raise RuntimeError('Primary comparison requires independent E3')
        if config['dct_spec']!=DCT_SPEC: raise RuntimeError('DCT contract mismatch')
        if config['normalization_mean']!=CELEBA_MEAN or config['normalization_std']!=CELEBA_STD: raise RuntimeError('RGB contract mismatch')
        if config['split_fingerprints']!=meta['split_fingerprints'] or config['bbox_cache_selected_fingerprint']!=meta['bbox_cache_selected_fingerprint']: raise RuntimeError('Metadata provenance conflict')
    else:
        if meta.get('backbone_name')!='MobileNetV3-Small' or meta.get('sample_seed')!=42: raise RuntimeError('Frozen E1 architecture/seed mismatch')
        if meta.get('train_samples')!= (98873 if mode=='preliminary' else meta.get('train_samples')): raise RuntimeError('E1 mode mismatch')
        if config.get('apply_gamma',config.get('research_apply_gamma',False)): raise RuntimeError('Gamma must be OFF')
        if config.get('mean',config.get('normalization_mean'))!=CELEBA_MEAN or config.get('std',config.get('normalization_std'))!=CELEBA_STD: raise RuntimeError('E1 normalization mismatch')
    if kind=='E1' and (config.get('bbox_expansion_factor')!=1.55 or config.get('model_img_size')!=224 or meta.get('bbox_expansion_factor')!=1.55): raise RuntimeError('Frozen E1 crop/input contract mismatch')
    session=cpu_session(onnx_path)
    ins=session.get_inputs()
    expected=[[3,224,224],[1,224,224]] if kind=='E3' else [[3,224,224]]
    if len(ins)!=len(expected) or any(i.type!='tensor(float)' or i.shape[1:]!=s for i,s in zip(ins,expected)): raise RuntimeError('ONNX input contract mismatch')
    if kind=='E3' and [i.name for i in ins]!=['rgb_input','frequency_map']: raise RuntimeError('E3 input names mismatch')
    if session.get_outputs()[0].shape[-1]!=3: raise RuntimeError('Output contract mismatch')
    return {'session':session,'threshold':thresholds[0],'config':config,'meta':meta,'onnx_path':onnx_path,'sha256':sha256_file(onnx_path),'onnx_sidecar_sha256':{p.name:sha256_file(p) for p in Path(onnx_path).parent.glob(Path(onnx_path).name+'.data*')}}

def infer_frozen(bundle,rgb,freq):
    ins=bundle['session'].get_inputs()
    vals=[rgb.numpy()[None].astype(np.float32),freq.numpy()[None].astype(np.float32)]
    logits=bundle['session'].run(None,{i.name:v for i,v in zip(ins,vals)})[0]
    if logits.shape!=(1,3) or not np.isfinite(logits).all(): raise RuntimeError('Invalid model inference')
    return logits[0]

## Resolve and audit manifests before inference
CSV: path,label,partition; video requires video_id; extracted frames also frame_index. Labels 0 Real / 1 Attack. Optional subject_id and attack_subtype. Paths relative to mounted root. Verify provenance before setting confirmation flag. Folder mapping applies only to image datasets.

In [6]:
models_frozen={name:load_frozen_model(name,directory,E1_RUN_MODE) for name,directory in [('E1',E1_ARTIFACT_DIR),('E3',E3_ARTIFACT_DIR)]}
e1_source=json.loads(Path(find_exact_artifact(f'mnv3s_e1_{E1_RUN_MODE}_v5_3_edge_run_config.json',E1_ARTIFACT_DIR)).read_text())
if e1_source['run_mode']!=E1_RUN_MODE: raise RuntimeError('Mixed E1 source mode')
for c in [models_frozen['E3']['config'],models_frozen['E3']['meta']]:
    if c['split_fingerprints']!=e1_source['split_fingerprints'] or c['bbox_cache_selected_fingerprint']!=e1_source['bbox_cache_selected_fingerprint']: raise RuntimeError('E1/E3 source protocols differ')


In [7]:
IMAGE_SUFFIXES={'.jpg','.jpeg','.png','.bmp','.webp'}
VIDEO_SUFFIXES={'.avi','.mp4','.mov','.mkv','.wmv','.m4v'}

def secure_path(root,rel):
    p=(Path(root)/str(rel)).resolve()
    if not p.is_relative_to(Path(root).resolve()): raise RuntimeError('Path escapes dataset root')
    if not p.is_file(): raise FileNotFoundError(p)
    return p

def resolve_dataset_manifest(name,root,cfg):
    if cfg['identity']!={'lcc_fasd':'LCC-FASD','casia_fasd':'CASIA-FASD','replay_attack':'Replay-Attack'}[name]: raise RuntimeError('Dataset identity mismatch (CASIA-SURF is not CASIA-FASD)')
    if not cfg['version'] or not cfg['source'] or not cfg['license'] or not cfg['provenance_confirmed']:
        raise RuntimeError(f'{name}: provenance/license unclear. Verify source/permission; fill version/source/license and explicitly set provenance_confirmed=True before inference.')
    if not cfg['partition']: raise RuntimeError('Specify target partition')
    kind=cfg['kind']
    if kind not in {'image','video','extracted_frames'}: raise ValueError('Unknown dataset kind')
    if cfg['manifest_csv']:
        frame=pd.read_csv(cfg['manifest_csv'],dtype={'path':str,'video_id':str,'subject_id':str,'partition':str})
        required={'path','label','partition'} | ({'video_id'} if kind!='image' else set()) | ({'frame_index'} if kind=='extracted_frames' else set())
        if not required<=set(frame.columns): raise RuntimeError(f'Required manifest columns: {required}')
        frame=frame[frame.partition==cfg['partition']].copy()
    else:
        if kind!='image': raise RuntimeError('Video/session adapter requires a CSV with explicit source video IDs and labels')
        rows=[]
        for folder,label in cfg['folder_mapping'].items():
            for p in sorted((Path(root)/folder).rglob('*')):
                if p.suffix.lower() in IMAGE_SUFFIXES:
                    rows.append({'path':str(p.relative_to(root)),'label':label,'partition':cfg['partition']})
        frame=pd.DataFrame(rows)
    if frame.empty: raise RuntimeError(f'No {name} samples: check partition and configured mapping')
    if frame.path.duplicated().any(): raise RuntimeError('Duplicate paths; mapping may overlap')
    if frame.label.isna().any() or not set(frame.label)<= {0,1}: raise RuntimeError('Labels must explicitly be 0 Real / 1 Attack')
    frame['label']=frame.label.astype(int)
    if set(frame.label)!={0,1}: raise RuntimeError('Target partition needs both Real and Attack')
    if kind=='image':
        if 'video_id' in frame and frame.video_id.notna().any(): raise RuntimeError('Image-only dataset must not invent video_id')
        frame['video_id']=''
    else:
        if frame.video_id.isna().any() or frame.video_id.eq('').any(): raise RuntimeError('Missing source video/session ID')
        for vid,g in frame.groupby('video_id'):
            if g.label.nunique()!=1: raise RuntimeError('Inconsistent labels within source video')
            if kind=='video' and len(g)!=1: raise RuntimeError('One source file per video ID required')
        if kind=='extracted_frames':
            if frame.frame_index.isna().any() or (frame.frame_index<0).any() or (frame.frame_index%1!=0).any(): raise RuntimeError('Frame indices must be nonnegative integers')
            if frame.duplicated(['video_id','frame_index']).any(): raise RuntimeError('Duplicate frame ID')
    frame['dataset']=name;frame['root']=str(Path(root).resolve());frame['kind']=kind
    frame['subject_id']=frame.get('subject_id','');frame['attack_subtype']=frame.get('attack_subtype','unknown')
    frame['file_sha256']=[sha256_file(secure_path(root,p)) for p in frame.path]
    return frame

def uniform_indices(n):
    if n<=0: return []
    return np.unique(np.linspace(0,n-1,min(FRAMES_PER_VIDEO,n),dtype=int)).tolist()

def sample_manifest(resolved):
    frames=[];units=[]
    for (dataset,kind),group in resolved.groupby(['dataset','kind'],sort=True):
        if kind=='image':
            for r in group.to_dict('records'):
                uid=r['path'];units.append({**r,'unit_id':uid,'candidate_frames':1})
                frames.append({**r,'unit_id':uid,'frame_index':0,'sample_status':'CANDIDATE'})
        elif kind=='video':
            for r in group.to_dict('records'):
                cap=cv2.VideoCapture(str(secure_path(r['root'],r['path'])));n=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                if not cap.isOpened() or n<=0:
                    cap.release();raise RuntimeError(f'Cannot obtain video frame count: {r["path"]}; prepare an explicit extracted-frame manifest')
                cap.release();indices=uniform_indices(n)
                units.append({**r,'unit_id':r['video_id'],'candidate_frames':len(indices),'total_source_frames':n})
                for index in indices: frames.append({**r,'unit_id':r['video_id'],'frame_index':index,'sample_status':'CANDIDATE'})
        else:
            for vid,g in group.groupby('video_id',sort=True):
                g=g.sort_values('frame_index');chosen=g.iloc[uniform_indices(len(g))]
                r=g.iloc[0].to_dict();units.append({**r,'unit_id':vid,'candidate_frames':len(chosen),'total_source_frames':len(g)})
                for r in chosen.to_dict('records'): frames.append({**r,'unit_id':vid,'sample_status':'CANDIDATE'})
    sampled=pd.DataFrame(frames);sampled['frame_id']=[hashlib.sha256(f'{r.dataset}|{r.path}|{int(r.frame_index)}'.encode()).hexdigest() for r in sampled.itertuples()]
    if sampled.frame_id.duplicated().any(): raise RuntimeError('Duplicate sampled frame IDs')
    return sampled,pd.DataFrame(units)

roots={'lcc_fasd':LCC_FASD_ROOT,'casia_fasd':CASIA_FASD_ROOT,'replay_attack':REPLAY_ATTACK_ROOT}
parts=[resolve_dataset_manifest(n,r,DATASET_CONFIG[n]) for n,r in roots.items() if r]
if not parts: raise RuntimeError('Configure LCC_FASD_ROOT and CASIA_FASD_ROOT')
resolved=pd.concat(parts,ignore_index=True);sampled,units=sample_manifest(resolved)
for name,g in units.groupby('dataset'):
    print({'dataset':name,'root':roots[name],'videos_sessions':len(g) if g.kind.iloc[0]!='image' else 0,'images':len(g) if g.kind.iloc[0]=='image' else None,'Real':int(g.label.eq(0).sum()),'Attack':int(g.label.eq(1).sum()),'attack_subtype_counts':g.loc[g.label==1,'attack_subtype'].value_counts().to_dict(),'candidate_frames':int(g.candidate_frames.sum())})
resolved.to_csv(RESULT_DIR/'external_dataset_manifest.csv',index=False)
save_json(RESULT_DIR/'external_dataset_manifest.json',resolved.to_dict('records'))
sampled.to_csv(RESULT_DIR/'sampled_frame_manifest.csv',index=False)
units.to_csv(RESULT_DIR/'evaluation_unit_manifest.csv',index=False)

{'dataset': 'lcc_fasd', 'root': '/kaggle/input/datasets/aleksandrpikul222/lcc-fasd/LCC_FASD', 'videos_sessions': 0, 'images': 3766, 'Real': 389, 'Attack': 3377, 'attack_subtype_counts': {'unknown': 3377}, 'candidate_frames': 3766}


## Deterministic external SCRFD cache and coverage
Largest valid detected face; 640→480→320 fallback; confidence .5; crop 1.55. No CelebA fallback. Failures remain in coverage. Cached results require identical file hashes, sampled manifest and detector policy.

In [8]:
# InsightFace loads the explicitly mounted SCRFD ONNX; no FaceAnalysis model download.
from insightface.model_zoo import model_zoo
if not SCRFD_MODEL_PATH or not Path(SCRFD_MODEL_PATH).is_file(): raise FileNotFoundError('Mount frozen det_500m.onnx and set SCRFD_MODEL_PATH')
if Path(SCRFD_MODEL_PATH).name!='det_500m.onnx': raise RuntimeError('Use frozen SCRFD-500M detector')
detector=model_zoo.get_model(SCRFD_MODEL_PATH,providers=['CPUExecutionProvider'])
detector.prepare(ctx_id=-1,input_size=(640,640),det_thresh=.5)
CACHE_POLICY={'schema':'external_scrfd_v1','detector_sha256':sha256_file(SCRFD_MODEL_PATH),'sizes':[[640,640],[480,480],[320,320]],'confidence':.5,'crop_factor':1.55,'selection':'largest valid face; confidence then xyxy tie-break','sampling':{'frames_per_video':16,'seed':42,'method':'uniform np.linspace integer indices'},'min_valid_frames':1,'opencv':cv2.__version__,'insightface':__import__('insightface').__version__}
manifest_hash=hashlib.sha256(sampled.to_csv(index=False).encode()).hexdigest()
external_cache={'policy':CACHE_POLICY,'sampled_manifest_sha256':manifest_hash,'records':{}}
if EXTERNAL_CACHE_INPUT_PATH:
    external_cache=json.loads(Path(EXTERNAL_CACHE_INPUT_PATH).read_text())
    if external_cache['policy']!=CACHE_POLICY or external_cache['sampled_manifest_sha256']!=manifest_hash: raise RuntimeError('External cache policy/manifest mismatch')
    if set(external_cache['records'])-set(sampled.frame_id): raise RuntimeError('Unexpected cached frame IDs')

def read_frame(r):
    path=secure_path(r['root'],r['path'])
    if r['kind']!='video': return cv2.imread(str(path))
    cap=cv2.VideoCapture(str(path));cap.set(cv2.CAP_PROP_POS_FRAMES,int(r['frame_index']));ok,image=cap.read();cap.release()
    return image if ok else None

def _scrfd_detect(detector, image_bgr, input_size):
    """
    Prefer detect(..., input_size=...). Older InsightFace variants are handled
    by temporarily changing detector.input_size.
    """
    try:
        return detector.detect(
            image_bgr,
            input_size=tuple(input_size),
            max_num=0,
            metric="default",
        )
    except TypeError:
        old_input_size = getattr(detector, "input_size", None)
        detector.input_size = tuple(input_size)
        try:
            return detector.detect(image_bgr, max_num=0, metric="default")
        finally:
            detector.input_size = old_input_size

def detect_record(r):
    rec={k:r[k] for k in ['dataset','video_id','frame_id','path','frame_index','unit_id','label']}
    image=read_frame(r)
    if image is None: return {**rec,'status':'READ_ERROR','n_faces':0,'reason':'Cannot decode candidate frame'}
    h,w=image.shape[:2];rec.update(image_width=w,image_height=h,crop_factor=1.55)
    for size in [(640,640),(480,480),(320,320)]:
        bboxes,_=_scrfd_detect(detector,image,size)
        valid=[]
        if bboxes is not None:
            for b in bboxes:
                x1,y1,x2,y2,conf=map(float,b[:5])
                if np.isfinite(b[:5]).all() and conf>=.5 and x2>x1 and y2>y1 and min(x2,w)>max(x1,0) and min(y2,h)>max(y1,0): valid.append([x1,y1,x2,y2,conf])
        if not valid: continue
        valid.sort(key=lambda b:(-(b[2]-b[0])*(b[3]-b[1]),-b[4],*b[:4]));b=valid[0]
        rec.update(bbox_xyxy=b[:4],confidence=b[4],det_input_size=list(size),n_faces=len(valid),ambiguous_multiface=len(valid)>1,face_min_side=min(b[2]-b[0],b[3]-b[1]),status='VALID')
        try: branch_inputs(cv2.cvtColor(runtime_crop_bgr(image,b[:4],1.55),cv2.COLOR_BGR2RGB))
        except (ValueError,RuntimeError,cv2.error) as exc: rec.update(status='PREPROCESS_ERROR',reason=str(exc))
        return rec
    return {**rec,'status':'NO_VALID_FACE','n_faces':0,'ambiguous_multiface':False}

for r in tqdm(sampled.to_dict('records'),desc='Build/reuse common external face cache'):
    if r['frame_id'] in external_cache['records']:
        cached=external_cache['records'][r['frame_id']]
        if any(cached.get(k)!=r[k] for k in ['frame_id','dataset','path','frame_index','unit_id','label']): raise RuntimeError('External cached record identity mismatch')
    if r['frame_id'] not in external_cache['records']:
        external_cache['records'][r['frame_id']]=detect_record(r)
        if len(external_cache['records'])%100==0: save_json(RESULT_DIR/'external_scrfd_cache.json',external_cache)
save_json(RESULT_DIR/'external_scrfd_cache.json',external_cache)
coverage=pd.DataFrame(list(external_cache['records'].values()))
for (name,label),g in coverage.groupby(['dataset','label']):
    print({'dataset':name,'label':label,'candidate_frames':len(g),'valid_frames':int(g.status.eq('VALID').sum()),'coverage':float(g.status.eq('VALID').mean()),'failure_count':int(g.status.ne('VALID').sum()),'multi_face_rate':float(g.get('n_faces',pd.Series(0,index=g.index)).gt(1).mean())})
coverage.to_csv(RESULT_DIR/'frame_coverage.csv',index=False)
print({'overall_candidate_frames':len(coverage),'overall_detector_coverage':float(coverage.status.eq('VALID').mean()),'overall_multi_face_rate':float(coverage.n_faces.gt(1).mean()),'overall_failures':int(coverage.status.ne('VALID').sum())})

Build/reuse common external face cache:   0%|          | 0/3766 [00:00<?, ?it/s]

{'dataset': 'lcc_fasd', 'label': np.int64(0), 'candidate_frames': 389, 'valid_frames': 389, 'coverage': 1.0, 'failure_count': 0, 'multi_face_rate': 0.0}
{'dataset': 'lcc_fasd', 'label': np.int64(1), 'candidate_frames': 3377, 'valid_frames': 3373, 'coverage': 0.9988155167308261, 'failure_count': 4, 'multi_face_rate': 0.0}
{'overall_candidate_frames': 3766, 'overall_detector_coverage': 0.9989378651088688, 'overall_multi_face_rate': 0.0, 'overall_failures': 4}


## Common valid frames; locked source thresholds; deterministic video mean d
Inference errors abort the comparison. Zero-valid-frame units retain NO_VALID_FACE and no prediction. Low-coverage videos remain included under MIN_VALID_FRAMES_PER_VIDEO=1. Undefined class metrics are null.

In [9]:
common_ids=set(coverage.loc[coverage.status=='VALID','frame_id'])
rows=[];inference_times={name:[] for name in models_frozen}
for r in tqdm(sampled.to_dict('records'),desc='E1 and E3 on common valid-face set'):
    if r['frame_id'] not in common_ids: continue
    rec=external_cache['records'][r['frame_id']];image=read_frame(r)
    if image is None: raise RuntimeError('Previously valid frame is unreadable; abort paired evaluation')
    crop=cv2.cvtColor(runtime_crop_bgr(image,rec['bbox_xyxy'],1.55),cv2.COLOR_BGR2RGB);rgb,freq=branch_inputs(crop)
    for name,bundle in models_frozen.items():
        start=time.perf_counter();logits=infer_frozen(bundle,rgb,freq);inference_times[name].append((time.perf_counter()-start)*1000)
        d=float(binary_pad_score_from_logits(logits[None])[0]);pred=int(d>=bundle['threshold'])
        rows.append({'model':name,'dataset':r['dataset'],'kind':r['kind'],'unit_id':r['unit_id'],'video_id':r['video_id'],'frame_id':r['frame_id'],'path':r['path'],'frame_index':r['frame_index'],'subject_id':r['subject_id'],'attack_subtype':r['attack_subtype'],'true_class':r['label'],'face_min_side':rec['face_min_side'],'image_width':rec['image_width'],'image_height':rec['image_height'],'crop_factor':1.55,'bbox_source':'external_SCRFD','logit_real':float(logits[0]),'logit_physical_spoof':float(logits[1]),'logit_digital_spoof':float(logits[2]),'d':d,'p_real':sigmoid(d),'pred_real':pred,'binary_correct':int(pred==(r['label']==0))})
frame_predictions=pd.DataFrame(rows,columns=['model','dataset','kind','unit_id','video_id','frame_id','path','frame_index','subject_id','attack_subtype','true_class','face_min_side','image_width','image_height','crop_factor','bbox_source','logit_real','logit_physical_spoof','logit_digital_spoof','d','p_real','pred_real','binary_correct'])
for name in models_frozen:
    if set(frame_predictions.loc[frame_predictions.model==name,'frame_id'])!=common_ids: raise RuntimeError('Paired comparison set mismatch')

# Preserve every candidate unit, including zero-valid-frame videos.
def aggregate_units(candidate_units,predictions,model_name,threshold):
    result=[]
    for r in candidate_units.to_dict('records'):
        g=predictions[(predictions.dataset==r['dataset'])&(predictions.unit_id==r['unit_id'])&(predictions.model==model_name)]
        count=len(g);valid=count>=MIN_VALID_FRAMES_PER_VIDEO
        d=float(g.d.mean()) if valid else None
        result.append({'model':model_name,'dataset':r['dataset'],'unit_id':r['unit_id'],'video_id':r['video_id'],'true_class':int(r['label']),'candidate_frames':int(r['candidate_frames']),'valid_frames':count,'coverage':count/r['candidate_frames'] if r['candidate_frames'] else 0.,'status':'VALID' if valid else 'NO_VALID_FACE','low_coverage':valid and count<r['candidate_frames'],'d':d,'pred_real':int(d>=threshold) if valid else None,'evaluation_unit':'image' if r['kind']=='image' else 'video','aggregation':'none' if r['kind']=='image' else 'arithmetic mean of valid sampled frame d'})
    return pd.DataFrame(result)

unit_predictions=pd.concat([aggregate_units(units,frame_predictions,n,b['threshold']) for n,b in models_frozen.items()],ignore_index=True)
unit_predictions.to_csv(RESULT_DIR/'unit_coverage_and_predictions.csv',index=False)
comparison=[];sizes=[];coverage_summary=[]
for name,group in units.groupby('dataset'):
    kind=group.kind.iloc[0];up=unit_predictions[unit_predictions.dataset==name];fp=frame_predictions[frame_predictions.dataset==name]
    if kind=='image': fp.to_csv(RESULT_DIR/f'{name}_image_predictions.csv',index=False)
    else:
        up.to_csv(RESULT_DIR/f'{name}_video_predictions.csv',index=False)
        fp.to_csv(RESULT_DIR/f'{name}_frame_predictions.csv',index=False)
    for model_name,bundle in models_frozen.items():
        all_units=up[up.model==model_name];valid_units=all_units[all_units.status=='VALID'];valid_frames=fp[fp.model==model_name]
        stats=pad_metrics(valid_units.true_class,valid_units.d,bundle['threshold'])
        comparison.append({'dataset':name,'model':model_name,'evaluation_unit':'image' if kind=='image' else 'video','candidate_units':len(all_units),'excluded_units':int(all_units.status.ne('VALID').sum()),'frame_detector_coverage':len(valid_frames)/int(group.candidate_frames.sum()),'unit_detector_coverage':len(valid_units)/len(all_units),'locked_logit_threshold':bundle['threshold'],**stats})
        sz=face_size_breakdown(valid_frames,bundle['threshold']);sz['dataset']=name;sz['model']=model_name;sz['evaluation_unit']='image' if kind=='image' else 'frame diagnostic';sizes.append(sz)
        save_plots(valid_units,bundle['threshold'],RESULT_DIR/f'{name}_{model_name}')
        for label,cg in all_units.groupby('true_class'):
            coverage_summary.append({'dataset':name,'model':model_name,'label':int(label),'candidate_units':len(cg),'evaluable_units':int(cg.status.eq('VALID').sum()),'excluded_units':int(cg.status.ne('VALID').sum()),'low_coverage_units':int(cg.low_coverage.sum()),'candidate_frames':int(cg.candidate_frames.sum()),'valid_frames':int(cg.valid_frames.sum())})
    # Side-by-side ROC/score distribution with each model's locked source threshold.
    fig,ax=plt.subplots()
    for m,g in up[up.status=='VALID'].groupby('model'):
        if g.true_class.eq(0).any() and g.true_class.ne(0).any():
            fpr,tpr,_=roc_curve(g.true_class.eq(0).astype(int),g.d);ax.plot(fpr,tpr,label=m)
    ax.set(xlabel='Attack acceptance rate',ylabel='Real acceptance rate');ax.legend()
    fig.savefig(RESULT_DIR/f'roc_{name}.png',bbox_inches='tight');plt.close(fig)
    fig,ax=plt.subplots()
    for m,g in up[up.status=='VALID'].groupby('model'):
        for real,cg in g.groupby(g.true_class.eq(0)):
            ax.hist(cg.d,bins=40,alpha=.3,label=f'{m} {"Real" if real else "Attack"}')
        ax.axvline(models_frozen[m]['threshold'],linestyle='--',label=f'{m} source threshold')
    ax.legend();ax.set_xlabel('d');fig.savefig(RESULT_DIR/f'score_distribution_{name}.png',bbox_inches='tight');plt.close(fig)
comparison=pd.DataFrame(comparison);comparison.to_csv(RESULT_DIR/'cross_dataset_model_comparison.csv',index=False)
pd.concat(sizes).to_csv(RESULT_DIR/'cross_dataset_face_size_breakdown.csv',index=False)
pd.DataFrame(coverage_summary).to_csv(RESULT_DIR/'coverage_by_class.csv',index=False)
deltas=[]
for name,g in comparison.groupby('dataset'):
    a=g.set_index('model')
    def delta(metric):
        e1,e3=a.loc['E1',metric],a.loc['E3',metric]
        return float(e3-e1) if pd.notna(e1) and pd.notna(e3) else None
    deltas.append({'dataset':name,'delta_AUC_E3_minus_E1':delta('AUC'),'delta_HTER_E3_minus_E1':delta('HTER'),'interpretation':'descriptive; no significance claim'})
# Optional in-domain display. Cross-unit CASIA comparison must not become a pure degradation delta.
in_domain=[]
for m,path in [('E1',E1_INDOMAIN_SUMMARY),('E3',E3_INDOMAIN_SUMMARY)]:
    if not path: continue
    source=json.loads(Path(path).read_text())
    sm=source.get('metrics',source)
    source_auc=sm.get('AUC',sm.get('auc'));source_error=sm.get('ACER',sm.get('acer'))
    if source_auc is not None and not 0<=source_auc<=1: raise RuntimeError('In-domain AUC must be a rate, not percent')
    if source_error is not None and not 0<=source_error<=1: raise RuntimeError('In-domain ACER must be a rate, not percent')
    for r in comparison[comparison.model==m].to_dict('records'):
        same_unit=r['evaluation_unit']=='image' and source.get('evaluation_unit')=='image'
        in_domain.append({'model':m,'target':r['dataset'],'source_AUC':source_auc,'source_ACER':source_error,'target_AUC':r['AUC'],'target_HTER':r['HTER'],'target_unit':r['evaluation_unit'],'caveat':'CelebA image vs video mean: units differ' if not same_unit else 'Image-level metrics; HTER=ACER under shared binary definitions','delta_AUC':r['AUC']-source_auc if same_unit and source_auc is not None and pd.notna(r['AUC']) else None,'delta_error':r['HTER']-source_error if same_unit and source_error is not None and pd.notna(r['HTER']) else None})
if in_domain: pd.DataFrame(in_domain).to_csv(RESULT_DIR/'in_domain_comparison.csv',index=False)
summary={'experiment':'Project cross-dataset / zero-shot stress test','source':'CelebA-Spoof','e1_run_mode':E1_RUN_MODE,'threshold_origin':'CelebA-Spoof Validation','target_threshold_tuning':False,'target_fine_tuning':False,'comparison':comparison.to_dict('records'),'deltas':deltas,'coverage':coverage_summary,'primary_targets_complete':{'lcc_fasd','casia_fasd'}<=set(units.dataset),'HTER_definition':'(FAR + FRR)/2; attack acceptance and real rejection; equals ACER here','conditional_PAD_metrics':'common successfully preprocessed frames; zero-valid-face units excluded and retained in coverage','face_size_note':'video face-size rows are frame diagnostics; native pixels do not measure common physical distance','in_domain_comparison':in_domain}
save_json(RESULT_DIR/'cross_dataset_summary.json',summary)
save_json(RESULT_DIR/'cross_dataset_protocol_snapshot.json',{'dataset_config':DATASET_CONFIG,'mounted_roots':roots,'available_license_metadata':{p:json.loads(Path(p).read_text()) for p in LICENSE_METADATA_PATHS},'models':{m:{'onnx_sha256':b['sha256'],'onnx_sidecar_sha256':b['onnx_sidecar_sha256'],'runtime_config':b['config'],'best_meta':b['meta']} for m,b in models_frozen.items()},'cache_policy':CACHE_POLICY,'sampled_manifest_sha256':manifest_hash,'provider':'CPUExecutionProvider','ort_threads':ORT_THREADS,'timing_diagnostic':{m:{'p50_ms':float(np.percentile(t,50)),'p95_ms':float(np.percentile(t,95)),'includes_preprocessing':False,'warmup':0,'sample_count':len(t),'note':'Observed inference diagnostic, not a paired controlled efficiency benchmark'} for m,t in inference_times.items() if t}})
zip_base=Path('/kaggle/working')/f'cross_dataset_{E1_RUN_MODE}_v1_results'
shutil.make_archive(str(zip_base),'zip',RESULT_DIR)
# Required convenient archive name, mode preserved in package metadata and folder.
shutil.copyfile(str(zip_base)+'.zip',Path('/kaggle/working')/'cross_dataset_e1_vs_e3_results.zip')
print(comparison.to_string(index=False));print('Download:',str(zip_base)+'.zip')

E1 and E3 on common valid-face set:   0%|          | 0/3766 [00:00<?, ?it/s]

 dataset model evaluation_unit  candidate_units  excluded_units  frame_detector_coverage  unit_detector_coverage  locked_logit_threshold  Accuracy    APCER    BPCER     ACER      AUC           confusion_matrix    N  N_real  N_attack     HTER
lcc_fasd    E1           image             3766               4                 0.998938                0.998938               -0.465022  0.614035 0.404388 0.226221 0.315304 0.753307  [[2009, 1364], [88, 301]] 3762     389      3373 0.315304
lcc_fasd    E3           image             3766               4                 0.998938                0.998938               -0.221487  0.655768 0.353395 0.264781 0.309088 0.744325 [[2181, 1192], [103, 286]] 3762     389      3373 0.309088
Download: /kaggle/working/cross_dataset_preliminary_v1_results.zip
